# TD02 — Réseau mono-couche (3 perceptrons)

# Objectif
Passer d'un neurone (TD01) à une **couche de 3 perceptrons** : `h = W·x + b`, `y = argmax(h)`, apprentissage de Rosenblatt multi-classes. On teste sur deux jeux 2D puis sur le jeu combiné, puis bonus lettres A/B/C en 5x5.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass, field

## 1. Deux jeux de données 2D
`gen_jeu1` : premier jeu (25 pts/classe par défaut).
`gen_jeu2` : second jeu (15 pts par défaut). Explorez leurs paramètres, puis visualisez.

In [ ]:
def gen_jeu1(n=25, sigma=0.6, seed=0):
    rng = np.random.default_rng(seed)
    centres = [(-2,-2),(2,-2),(0,2)]
    X_list, y_list = [], []
    for k,(cx,cy) in enumerate(centres):
        pts = rng.normal(loc=(cx,cy), scale=sigma, size=(n,2))
        X_list.append(pts)
        y_list += [k]*n
    return np.vstack(X_list), np.array(y_list)

def gen_jeu2(n=15, seed=1):
    rng = np.random.default_rng(seed)
    Xa = rng.normal(loc=(0,0), scale=0.6, size=(n,2))
    Xb = rng.normal(loc=(0,0), scale=0.6, size=(n,2))
    X = np.vstack([Xa, Xb])
    y = np.array([0]*n + [1]*n)
    return X, y

## 2. Construction des jeux d'entraînement
`X1` : premier jeu seul. `X_comb` : combinaison des deux jeux (`X1 + X2`) pour tester les limites du modèle mono-couche.

In [ ]:
X1, y1 = gen_jeu1()
X2, y2 = gen_jeu2()
X_comb = np.vstack([X1, X2])
y_comb = np.concatenate([y1, y2])

## 3. Outil de visualisation `display_data`
Nuage coloré par classe, aspect égal, grille. Visualisez chaque jeu avant d'entraîner.

In [ ]:
def display_data(X, y, titre=None):
    couleurs = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red', 'tab:purple']
    fig, ax = plt.subplots()
    for k in sorted(set(y)):
        ax.scatter(
            X[y == k, 0], X[y == k, 1],
            c=couleurs[k % len(couleurs)],
            edgecolors='k', s=25, label=f"classe {k}"
        )
    ax.set_aspect('equal')
    ax.set_xlabel("x1")
    ax.set_ylabel("x2")
    if titre:
        ax.set_title(f"{titre} ({len(X)} pts)")
    ax.legend()
    ax.grid(True)
    plt.show()

### 3.1 Jeu 1 vs jeu 2
Ci-dessous : `X1` puis `X2`. Que constatez-vous ?

## 4. Modèle : couche mono-couche `y = argmax(Wx + b)`
`W : (K×D)`, `b : (K,)`. Règle de Rosenblatt multi-classes (Exercice 1) : si `pred != target`, on **renforce** le bon neurone `W[t] += r·x` et on **punit** le mauvais gagnant `W[p] -= r·x` (idem biais). `train` répète au plus 50 époques, arrêt précoce si 0 erreur.

In [ ]:
from dataclasses import dataclass
import numpy as np

@dataclass
class CoucheMonoCouche:
    W: np.ndarray
    b: np.ndarray
# TODO


## 5. Entraînement : jeu 1 vs jeu combiné
Deux couches identiques (Exercices 2-3) : `couche1` sur `X1`, `couche_comb` sur `X_comb`. Comparez les accuracies : que se passe-t-il ?

In [ ]:
couche1 = CoucheMonoCouche(W=np.zeros((3,2)), b=np.zeros(3))
couche1.train(X1, y1)
pred1 = np.array([couche1.predict(x) for x in X1])
print(f"accuracy jeu 1: {(pred1==y1).mean()*100:.1f}%")
couche_comb = CoucheMonoCouche(W=np.zeros((3,2)), b=np.zeros(3))
couche_comb.train(X_comb, y_comb)
pred_comb = np.array([couche_comb.predict(x) for x in X_comb])
print(f"accuracy jeu combine: {(pred_comb==y_comb).mean()*100:.1f}%")

### 5.1 Frontieres de decision avec `plot_frontieres`
- **Fond colore** = vraie decision `argmax_k h_k` (3 regions convexes, bords = `h_i == h_j`).
- **Pointilles noirs** = zeros de chaque score `h_k = 0` (ou le neurone k bascule +/-). **Ce N'EST PAS la frontiere de decision !**
- La legende `h_k=0 (indicatif)` le rappelle.

> [!faq] Exercice 2-3 — Lire la figure
> 1. Trouvez un pointille qui traverse une zone de couleur uniforme. Que concluez-vous ?
> 2. A gauche (jeu 1) : les 3 regions couvrent bien les 3 blobs → 100%.
> 3. A droite (jeu combine) : le piege central (classes 0/1 melangees en (0,0)) tombe dans une seule region → <100%, oscillation. Lien avec XOR du TD01 ? Pourquoi aucun `W` ne marche ?

In [ ]:
def plot_frontieres(ax, X, y, couche, lim=(-4,4)):
    xx, yy = np.meshgrid(np.linspace(lim[0],lim[1],300), np.linspace(lim[0],lim[1],300))
    grid = np.c_[xx.ravel(), yy.ravel()]
    Z = np.argmax(grid @ couche.W.T + couche.b, axis=1).reshape(xx.shape)
    # fond = vraie decision argmax (bords = h_i == h_j)
    ax.contourf(xx, yy, Z, levels=[-0.5,0.5,1.5,2.5], colors=['tab:blue','tab:orange','tab:green'], alpha=0.12)
    for k,c in enumerate(['tab:blue','tab:orange','tab:green']):
        ax.scatter(X[y==k,0], X[y==k,1], c=c, edgecolors='k', s=25, label=f"classe {k}")
    # pointilles h_k=0 : indicatif uniquement, PAS la frontiere argmax
    for k in range(3):
        H = (grid @ couche.W[k] + couche.b[k]).reshape(xx.shape)
        ax.contour(xx, yy, H, levels=[0], colors='k', linewidths=1.5, linestyles='--')
    ax.plot([], [], 'k--', label='h_k=0 (indicatif, pas frontiere)')
    ax.set_aspect('equal'); ax.legend(fontsize=8); ax.grid(True)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11,4.5))
plot_frontieres(axes[0], X1, y1, couche1)
axes[0].set_title("Jeu 1")
plot_frontieres(axes[1], X_comb, y_comb, couche_comb)
axes[1].set_title("Jeu combine")
plt.tight_layout()
plt.show()

## 6. Bonus actif : lettres A/B/C en 5x5, 25D (Exercices 4-6)
Meme classe `CoucheMonoCouche` reutilisee en dimension 25 : 3 templates `TPL` encodes en `±1`, bruites par `n_flip` pixels inverses.
Objectifs : (1) montrer que le meme modele marche en 25D quand c'est separable (contraste avec le piege central), (2) lire chaque ligne de `W` comme un prototype A/B/C.

In [ ]:
TPL = {
 0: np.array([0,1,0,1,0, 1,0,1,0,1, 1,1,1,1,1, 1,0,0,0,1, 1,0,0,0,1]),
 1: np.array([1,1,1,0,0, 1,0,0,1,0, 1,1,1,0,0, 1,0,0,1,0, 1,1,1,0,0]),
 2: np.array([0,1,1,1,0, 1,0,0,0,0, 1,0,0,0,0, 1,0,0,0,0, 0,1,1,1,0]),
}
# TODO


### 6.1 Entrainement + comprendre les scores `h`
Train avec `n_flip=2` → ~100%. Puis on affiche **une** lettre bruitee + ses 3 scores `h = W.x + b` : le gagnant `argmax` est la prediction.

> [!faq] Exercice 4 — Predire a la main
> Choisissez une lettre de test, affichez-la en 5x5, lisez ses 3 scores. Quel `h_k` est le plus grand ? Retrouvez la prediction `argmax`.

In [ ]:
X_let, y_let = gen_lettres()
c_let = CoucheMonoCouche(W=np.zeros((3,25)), b=np.zeros(3))
c_let.train(X_let, y_let)
pred_let = np.array([c_let.predict(x) for x in X_let])
print(f"accuracy lettres train: {(pred_let==y_let).mean()*100:.1f}%")
X_let_test, y_let_test = gen_lettres(n_per=10, n_flip=5, seed=99)
pred_test = np.array([c_let.predict(x) for x in X_let_test])
print(f"accuracy lettres test flip5: {(pred_test==y_let_test).mean()*100:.1f}%")

def show_lettre(x, couche, titre=None):
    h = couche.W @ x + couche.b
    fig, axes = plt.subplots(1, 2, figsize=(6,2.5))
    axes[0].imshow(x.reshape(5,5), cmap="RdBu", vmin=-1, vmax=1)
    axes[0].set_title(titre or "lettre 5x5 (noir=-1, rouge=+1)")
    axes[0].axis("off")
    axes[1].bar([0,1,2], h, color=['tab:blue','tab:orange','tab:green'])
    axes[1].set_xticks([0,1,2]); axes[1].set_xticklabels(['h0(A)','h1(B)','h2(C)'])
    axes[1].set_title(f"scores h, pred={int(np.argmax(h))}")
    axes[1].grid(True, axis='y')
    plt.tight_layout(); plt.show()
    return h

In [ ]:
show_lettre(X_let_test[0], c_let, titre=f"test0 vrai={y_let_test[0]} ({'ABC'[y_let_test[0]]}), pred={c_let.predict(X_let_test[0])}")

### 6.2 Acte 1 — Lire `W` : quel prototype pour quelle lettre ?
Chaque ligne de `W` (25 nombres) re-affichee en 5x5 (`RdBu`, rouge=+1, bleu=-1) : le reseau a appris un prototype.

> [!faq] Exercice 5 — Associer
> 1. Affichez haut=templates `TPL`, bas=`W` appris. Associez chaque `W_k` a A/B/C et justifiez (quels pixels discriminent B vs C ? barre centrale ?).
> 2. Pourquoi `W` ressemble au template mais en flou / valeurs intermediaires ? (moyenne des exemples + corrections Rosenblatt).

In [ ]:
# TODO


### 6.3 Acte 2 — Courbe de robustesse au bruit (a coder)
On fait varier `n_flip` de 0 a 8 : train fixe (`n_flip=2`), test de plus en plus bruite. Tracez accuracy train vs test.
Attendu : 100% a flip 0-2, ~70% a flip 5, ~60% a flip 8. C'est une mesure de generalisation au bruit (lien partie 3 §4 surapprentissage).

> [!faq] Exercice 6 — Robustesse
> Completez la boucle `for nf in flips` + le `plt.plot`. A partir de quel `n_flip` le modele passe sous 80% ? Pourquoi ? (distance au prototype vs bruit).

In [ ]:
# TODO


### 6.4 Acte 3 — Synthese : dimension vs separabilite

> [!danger] Exercice 7 — Synthese
> Pourquoi le meme modele echoue sur le piege central 2D mais reussit en 25D sur les lettres ?
> Reponse attendue : ce n'est pas la dimension qui compte, c'est la **separabilite lineaire**. Lettres = 3 nuages bien separes autour des prototypes (±1 + quelques flips) → Rosenblatt converge. Piege = classes 0/1 melangees au centre → aucun `W` lineaire ne marche, il faut du non-lineaire / multi-couches (TD03).

## Conclusion
- Separable → Rosenblatt converge, regions convexes (fond `argmax`).
- `h_k=0` (pointilles) ≠ frontiere : la frontiere est `h_i == h_j`.
- Piege central → oscillation, <100% : limite du lineaire.
- Lettres 25D → OK car separable ; `W` lisible = prototypes ; robustesse chute avec le bruit.
- Suite : besoin de non-linearite / multi-couches (cf. TD03).